# RAPIDD verification for germanium

Use **Run All** to explore the saved comparison. Set **RUN_LIVE** to rerun the same analysis against the pinned, unmodified RAPIDD C source or to change the kernel grid. Both modes plot the results without overwriting the saved comparison.

Run with a Python kernel that has NumPy, SciPy and Matplotlib installed. For the live mode, also install Git and GCC (on Windows, a WSL Python kernel is one option). From the WIMP-LEGEND repository root:

~~~sh
git clone https://github.com/cheekyparticle/RAPIDD_for_DM ../RAPIDD_for_DM
git -C ../RAPIDD_for_DM checkout c36b290abfc8f68dc4e53445bc18c185f4b466b0
~~~

The comparison covers the 14 diagonal, single-operator kernels of the frozen basis: O₁ and O₃–O₁₅. The original three operators were the poster workbench's single-operator benchmark. Off-diagonal operator interference, absolute halo-integrated rates, and detector-folded limits are separate checks. O₂ is outside the adopted basis.

[Comparison report](../docs/RAPIDD_COMPARISON.md) · [Conventions](../docs/NREFT_CONVENTIONS.md)

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

# Start Jupyter from this checkout or from its notebooks/ directory.
cwd = Path.cwd().resolve()
ROOT = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "analyses/validation/rapidd/compare_kernels.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the WIMP-LEGEND checkout.")

VALIDATION = ROOT / "analyses/validation/rapidd"
RUN_LIVE = False
RAPIDD_CHECKOUT = ROOT.parent / "RAPIDD_for_DM"

# Edit the grid only in live mode. Couplings are dimensionful (GeV^-2).
OPERATORS = (1, *range(3, 16))
ISOTOPES = (70, 72, 73, 74, 76)
RECOILS_KEV = (1.0, 10.0, 30.0)
MASSES_GEV = (40.0,)
SPEEDS_KM_S = (700.0,)
COUPLINGS = ((1.0, 1.0), (1.0, 0.0), (1.0, -0.7), (0.0, 1.0))

In [ ]:
if RUN_LIVE:
    for path in (ROOT / "src", VALIDATION):
        if str(path) not in sys.path:
            sys.path.insert(0, str(path))

    import compare_kernels
    import compare_responses

    kernels = compare_kernels.run(
        RAPIDD_CHECKOUT,
        operators=OPERATORS,
        isotopes=ISOTOPES,
        energies_keV=RECOILS_KEV,
        masses_GeV=MASSES_GEV,
        speeds_km_s=SPEEDS_KM_S,
        couplings=COUPLINGS,
    )
    responses = compare_responses.run(RAPIDD_CHECKOUT, include_phi_candidate=True)
else:
    saved = json.loads((VALIDATION / "all_operator_comparison.json").read_text())
    for report in saved.values():
        report["records"] = [
            dict(zip(report["columns"], row, strict=True)) for row in report["rows"]
        ]
    kernels, responses = saved["kernels"], saved["responses"]

assert kernels["rapidd_commit"] == responses["rapidd_commit"]
print("RAPIDD revision:", kernels["rapidd_commit"])
print("Kernel points:", len(kernels["records"]))
print("Nuclear-input points:", len(responses["records"]))
skipped = kernels.get("skipped_kinematic_points", [])
if skipped:
    print("Skipped forbidden (isotope, recoil, mass, speed) points:", len(skipped))
    print("First skipped point:", skipped[0])

## Which velocity enters the operator?

At fixed elastic recoil, the incoming velocity component along the momentum transfer has magnitude \(v_{\min}=q/(2\mu_T)\). Removing it leaves \(v_{T,\perp}^2=v^2-v_{\min}^2\). The parallel component supplies the recoil, so at \(v=v_{\min}\) the transverse component vanishes.

Write the two RAPIDD form-factor terms as \(F_0\) and \(F_2\). Its pinned rate assembly weights them with the raw inverse-speed and speed moments, equivalent at fixed speed to \(F_0+v^2 F_2\). The diagnostic evaluates \(F_0+(v^2-v_{\min}^2)F_2\). At the halo-integral level the equivalent change is \(\xi-v_{\min}^2\eta\) multiplying \(F_2\), where \(\xi=\int_{v>v_{\min}}d^3v\,f(\mathbf v)v\) and \(\eta=\int_{v>v_{\min}}d^3v\,f(\mathbf v)/v\).

The shared-input comparison isolates the apparent omission to the pinned RAPIDD assembly. Upstream confirmation and full rate-level tests remain open. The diagnostic leaves both implementations' production equations as shipped. For O₈ in even Ge, the \(\Delta\) contribution vanishes, so the relative excess is \(v_{\min}^2/(v^2-v_{\min}^2)\).

[RAPIDD rate assembly](https://github.com/cheekyparticle/RAPIDD_for_DM/blob/c36b290abfc8f68dc4e53445bc18c185f4b466b0/lib/source/difRateGen.c#L112-L127) · [Halo moments](https://github.com/cheekyparticle/RAPIDD_for_DM/blob/c36b290abfc8f68dc4e53445bc18c185f4b466b0/lib/source/halo.c#L95-L154)

In [ ]:
def max_abs_relative(rows, field):
    return max((abs(row[field]) for row in rows if row[field] is not None), default=np.nan)


raw_key = "rapidd_raw_velocity_kernel"
shifted_key = "shifted_velocity_diagnostic"
columns = ("raw_relative_difference", "shifted_relative_difference")
operators = sorted({row["operator"] for row in kernels["records"]})
kernel_maxima = []
print(f"{'Operator':>8} {'Raw max':>12} {'Shifted max':>12} {'Common zeros':>14} {'Zero mismatch':>14}")
for op in operators:
    rows = [row for row in kernels["records"] if row["operator"] == op]
    maxima = [max_abs_relative(rows, field) for field in columns]
    common_zeros = sum(row["ours"] == row[raw_key] == row[shifted_key] == 0 for row in rows)
    zero_mismatches = sum(
        (row["ours"] == 0) != (row[shifted_key] == 0) for row in rows
    )
    kernel_maxima.append(maxima)
    print(f"O{op:<7} {maxima[0]:12.3e} {maxima[1]:12.3e} {common_zeros:14d} {zero_mismatches:14d}")

if np.any(np.asarray(kernel_maxima) > 0):
    fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
    for values, marker, label in zip(
        np.asarray(kernel_maxima).T, ("o", "x"), ("RAPIDD raw", "Transverse-velocity diagnostic")
    ):
        ax.scatter(operators, np.where(values > 0, values, np.nan), marker=marker, label=label)
    ax.set(yscale="log", ylabel="Maximum absolute relative difference",
           xticks=operators, xticklabels=[f"O{op}" for op in operators])
    ax.legend()
    plt.show()
elif operators:
    print("No positive relative differences to show on the log scale.")
else:
    print("No allowed kernel points in this grid.")

## Inspect a recoil slice

Change the isotope, operator and coupling pair below. The numerator is the C kernel and the denominator is WIMP-LEGEND on the same nuclear inputs. Exact-zero denominators are reported in the table above. For O₁₂ and O₁₃ the shared fixture uses RAPIDD's \(Pp/Pp+Pt/Pt\) combination as the diagnostic \(\tilde{\Phi}'\) response.

In [ ]:
A_PLOT, OP_PLOT = 76, 8
MASS_PLOT_GEV, SPEED_PLOT_KM_S = 40.0, 700.0
CP_PLOT, CN_PLOT = 1.0, 1.0

slice_rows = sorted(
    (r for r in kernels["records"] if
     (r["A"], r["operator"], r["mchi_GeV"], r["v_km_s"], r["cp"], r["cn"])
     == (A_PLOT, OP_PLOT, MASS_PLOT_GEV, SPEED_PLOT_KM_S, CP_PLOT, CN_PLOT)),
    key=lambda r: r["ER_keV"],
)
if slice_rows:
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4), layout="constrained")
    recoil = [row["ER_keV"] for row in slice_rows]
    for field, label in (("ours", "WIMP-LEGEND"), (raw_key, "RAPIDD raw"),
                         (shifted_key, "Transverse-velocity diagnostic")):
        left.plot(recoil, [row[field] for row in slice_rows], "o--", label=label)
    for field, label in zip(columns, ("RAPIDD raw", "Transverse-velocity diagnostic")):
        values = np.array([row[field] for row in slice_rows], dtype=float)
        right.plot(recoil, 100 * values, "o--", label=label)
    left.set(ylabel=r"Kernel [GeV$^{-4}$]")
    right.set(ylabel="Relative difference [%]")
    for ax in (left, right):
        ax.set(xlabel=r"$E_R$ [keV]")
        ax.legend(fontsize="small")
    fig.suptitle(f"Ge-{A_PLOT}, O{OP_PLOT}, {MASS_PLOT_GEV:g} GeV, {SPEED_PLOT_KM_S:g} km/s")
    plt.show()
else:
    print("This slice has no points. Set the controls to values present in the grid.")

## Compare nuclear inputs at matching \(y=(qb/2)^2\)

Here each code uses its own nuclear input. The original comparison omitted \(\tilde{\Phi}'\). The source contains \(Pp/Pp\) entries for Ge-73 and the O₁₂/O₁₃ routines sum them with \(Pt/Pt\). The candidate mapping is plotted as \(\tilde{\Phi}'\) below. Shared-input kernel agreement for those operators checks the contraction. The size and origin of the nuclear-input difference remain a separate question.

[RAPIDD Ge-73 table](https://github.com/cheekyparticle/RAPIDD_for_DM/blob/c36b290abfc8f68dc4e53445bc18c185f4b466b0/lib/source/effFormFact.c#L339-L342)

In [ ]:
channel_labels = {
    "M": r"$M$",
    "Sigma_prime": r"$\Sigma'$",
    "Sigma_double_prime": r"$\Sigma''$",
    "Phi_double_prime": r"$\Phi''$",
    "Phi_double_prime_M": r"$\Phi''M$",
    "Phi_tilde_prime": r"$\widetilde{\Phi}'$ candidate",
    "Delta": r"$\Delta$",
    "Delta_Sigma_prime": r"$\Delta\Sigma'$",
}
response_maxima = {
    channel: max_abs_relative(
        (r for r in responses["records"] if r["channel"] == channel),
        "relative_difference",
    )
    for channel in channel_labels
}
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.barh(list(channel_labels.values()), list(response_maxima.values()))
ax.set(xscale="log", xlabel="Maximum absolute relative difference at matched y")
ax.invert_yaxis()
plt.show()

zero_mismatches = [
    row for row in responses["records"]
    if (row["rapidd"] == 0) != (row["afh_source"] == 0)
]
print("Nuclear-input zero mismatches:", len(zero_mismatches))
phi_rows = [
    row for row in responses["records"]
    if row["A"] == 73 and row["channel"] == "Phi_tilde_prime"
    and row["n1"] == row["n2"] == 1
]
print("Ge-73 candidate response (neutron–neutron)")
for row in phi_rows:
    print(f"y={row['y']:g}   RAPIDD/AFH={1 + row['relative_difference']:.5g}")